In [1]:
from QLBM import QLBM, collision, InitializeQC
from QLBM_1 import QLBM1, collision1, InitializeQC1
from QLBM_2 import QLBM2, collision2, InitializeQC2
from QLBM_3 import QLBM3, collision3, InitializeQC3
import numpy as np
import matplotlib.pyplot as plt
import qiskit_aer
from qiskit import transpile

In [ ]:
# Domain and grid setup
N_POINTS_X, N_POINTS_Y, N_POINTS_Z = 32, 32, 32
# Simulation parameters

NUMBER_DISCRETE_VELOCITIES = 27  # D2Q9 lattice configuration

TIMESTEPS = 40000

In [ ]:
Qq = 27
Nx = N_POINTS_X-1
Ny = N_POINTS_Y-1
Nz = N_POINTS_Y-1
dx = 1
dy = 1
dz = 1
dt = dx
c = dt/dx
Re = 100
f_eq = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
f = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
f_star = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
Lx = dx * float(Nx+1)
Ly = dy * float(Ny+1)
Lz = dz * float(Nz+1)
U = 0.1
nu = U*Lx/Re
cs = np.sqrt(c**2/3)
tau_f = 1.0
rho_0 = 1.0
rho = np.zeros((Nx+1, Ny+1, Ny+1))
u = np.zeros((Nx+1, Ny+1, Ny+1, 3))
e = np.array([
    [0,  1, -1,  0,  0,   0,  0,  1, -1,  1, -1,  0,  0,  1, -1,  1, -1,  0,  0,  1, -1,  1, -1,  1, -1, -1,  1],
    [0,  0,  0,  1, -1,   0,  0,  1, -1,  0,  0,  1, -1, -1,  1,  0,  0,  1, -1,  1, -1,  1, -1, -1,  1,  1, -1],
    [0,  0,  0,  0,  0,   1, -1,  0,  0,  1, -1,  1, -1,  0,  0, -1,  1, -1,  1,  1, -1, -1,  1,  1, -1,  1, -1]
]).T
w = np.array([
    8/27,                   
    2/27, 2/27, 2/27, 2/27, 2/27, 2/27,  
    1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54,  
    1/216, 1/216, 1/216, 1/216, 1/216, 1/216, 1/216, 1/216  
])

A = 3/4 - 9/2*U*Lx/(Re*dt)
nu1 = (1/6-2/9*A)*dt
u_n = u[:, :, :, 0].copy()
v_n = u[:, :, :, 1].copy() 
w_n = u[:, :, :, 2].copy()
u_t = np.zeros((Nx+3, Ny+3, Ny+3))
v_t = np.zeros((Nx+3, Ny+3, Ny+3))
w_t = np.zeros((Nx+3, Ny+3, Ny+3))
print(nu,nu1,Lx,Ly,TIMESTEPS)

q_error=[]
c_error=[]
qc_error=[]

(27, 3)
0.032 0.032 32.0 32.0 4000


In [ ]:
## Initial
x0 = np.linspace(0,Lx,Nx+1)
y0 = np.linspace(0,Ly,Ny+1)
z0 = np.linspace(0,Lz,Nz+1)
print(x0,y0,z0)
X0,Y0,Z0 = np.meshgrid(x0,y0,z0)
rho[:, :, :] = rho_0
u[:, :, Nz, 0] = U
u[:, :, :, 1] = 0
u[:, :, :, 2] = 0
print(rho.shape,u.shape,X0.shape,U,rho_0)

[ 0.          1.03225806  2.06451613  3.09677419  4.12903226  5.16129032
  6.19354839  7.22580645  8.25806452  9.29032258 10.32258065 11.35483871
 12.38709677 13.41935484 14.4516129  15.48387097 16.51612903 17.5483871
 18.58064516 19.61290323 20.64516129 21.67741935 22.70967742 23.74193548
 24.77419355 25.80645161 26.83870968 27.87096774 28.90322581 29.93548387
 30.96774194 32.        ] [ 0.          1.03225806  2.06451613  3.09677419  4.12903226  5.16129032
  6.19354839  7.22580645  8.25806452  9.29032258 10.32258065 11.35483871
 12.38709677 13.41935484 14.4516129  15.48387097 16.51612903 17.5483871
 18.58064516 19.61290323 20.64516129 21.67741935 22.70967742 23.74193548
 24.77419355 25.80645161 26.83870968 27.87096774 28.90322581 29.93548387
 30.96774194 32.        ] [ 0.          1.03225806  2.06451613  3.09677419  4.12903226  5.16129032
  6.19354839  7.22580645  8.25806452  9.29032258 10.32258065 11.35483871
 12.38709677 13.41935484 14.4516129  15.48387097 16.51612903 17.5483871
 1

In [8]:
simulator = qiskit_aer.backends.statevector_simulator.StatevectorSimulator()
simulator1 = qiskit_aer.backends.statevector_simulator.StatevectorSimulator()
simulator2 = qiskit_aer.backends.statevector_simulator.StatevectorSimulator()
simulator3 = qiskit_aer.backends.statevector_simulator.StatevectorSimulator()

In [ ]:
# Initialize the quantum LBM scalar field
Psi_qlbm = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm1 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm2 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm3 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm[0, :, :, :] = rho_0#Psi_init
Psi_qlbm1[0, :, :, :] = u[:,:,:,0].copy()#Psi_init
Psi_qlbm2[0, :, :, :] = u[:,:,:,1].copy()#Psi_init
Psi_qlbm3[0, :, :, :] = u[:,:,:,2].copy()#Psi_init
Psi_qlbm0 = Psi_qlbm[0,:,:,:].copy()
u_LBM = np.zeros((N_POINTS_X, N_POINTS_Y, N_POINTS_Z, 3))
u_LBM[:, :, :, 0] = Psi_qlbm1[0, :, :,:]  # Set the x-component of the velocity
u_LBM[:, :, :, 1] = Psi_qlbm2[0, :, :,:]  # Set the y-component of the velocity
u_LBM[:, :, :, 2] = Psi_qlbm3[0, :, :,:]  # Set the z-component of the velocity

# Quantum LBM simulation loop
for t in range(TIMESTEPS):
    u_t[1:-1,1:-1,1:-1] = u[:,:,:,0].copy()
    v_t[1:-1,1:-1,1:-1] = u[:,:,:,1].copy()
    w_t[1:-1,1:-1,1:-1] = u[:,:,:,2].copy()

    u_t[1:-1,1:-1,1:-1] = u[:,:,:,0].copy()
    v_t[1:-1,1:-1,1:-1] = u[:,:,:,1].copy()
    w_t[1:-1,1:-1,1:-1] = u[:,:,:,2].copy()

    u_t[0,1:-1,1:-1] = -u[1,:,:,0].copy()
    u_t[-1,1:-1,1:-1] = -u[-2,:,:,0].copy()
    u_t[1:-1,0,1:-1] = -u[:,1,:,0].copy()
    u_t[1:-1,-1,1:-1] = -u[:,-2,:,0].copy()
    u_t[1:-1,1:-1,0] = -u[:,:,1,0].copy()
    u_t[1:-1,1:-1,-1] = 2.0*U-u[:,:,-2,0].copy()

    v_t[0,1:-1,1:-1] = -u[1,:,:,1].copy()
    v_t[-1,1:-1,1:-1] = -u[-2,:,:,1].copy()
    v_t[1:-1,0,1:-1] = -u[:,1,:,1].copy()
    v_t[1:-1,-1,1:-1] = -u[:,-2,:,1].copy()
    v_t[1:-1,1:-1,0] = -u[:,:,1,1].copy()
    v_t[1:-1,1:-1,-1] = -u[:,:,-2,1].copy()

    w_t[0,1:-1,1:-1] = -u[1,:,:,2].copy()
    w_t[-1,1:-1,1:-1] = -u[-2,:,:,2].copy()
    w_t[1:-1,0,1:-1] = -u[:,1,:,2].copy()
    w_t[1:-1,-1,1:-1] = -u[:,-2,:,2].copy()
    w_t[1:-1,1:-1,0] = -u[:,:,1,2].copy()
    w_t[1:-1,1:-1,-1] = -u[:,:,-2,2].copy()


    u_t_y = (u_t[1:-1, 2:, 1:-1] - u_t[1:-1, :-2, 1:-1])/(2.0*dy)
    u_t_x = (u_t[2:, 1:-1, 1:-1] - u_t[:-2, 1:-1, 1:-1])/(2.0*dx)
    u_t_z = (u_t[1:-1, 1:-1, 2:] - u_t[1:-1, 1:-1, :-2])/(2.0*dz)
   
    v_t_y = (v_t[1:-1, 2:, 1:-1] - v_t[1:-1, :-2, 1:-1])/(2.0*dy)
    v_t_x = (v_t[2:, 1:-1, 1:-1] - v_t[:-2, 1:-1, 1:-1])/(2.0*dx)
    v_t_z = (v_t[1:-1, 1:-1, 2:] - v_t[1:-1, 1:-1, :-2])/(2.0*dz)

    w_t_y = (w_t[1:-1, 2:, 1:-1] - w_t[1:-1, :-2, 1:-1])/(2.0*dy)
    w_t_x = (w_t[2:, 1:-1, 1:-1] - w_t[:-2, 1:-1, 1:-1])/(2.0*dx)
    w_t_z = (w_t[1:-1, 1:-1, 2:] - w_t[1:-1, 1:-1, :-2])/(2.0*dz)

    temp = u[:, :, :, 0] * u[:, :, :, 0] + u[:, :, :, 1] * u[:, :, :, 1]+ u[:, :, :, 2] * u[:, :, :, 2]
    for Q in range(27):
        f_eq[Q,:, :, :] = w[Q]*rho*(1.0 + 3.0 * (e[Q, 0] * u[:, :, :, 0] + e[Q, 1] * u[:, :, :, 1] + e[Q, 2] * u[:, :, :, 2]) + 4.5 * (e[Q, 0] * u[:, :, :, 0] + e[Q, 1] * u[:, :, :, 1] + e[Q, 2] * u[:, :, :, 2]) ** 2 - 1.5 * temp +\
                                  A*dt*(u_t_x + u_t_x)*e[Q,0]*e[Q,0] + A*dt*(v_t_x + u_t_y)*e[Q,0]*e[Q,1] + A*dt*(w_t_x + u_t_z)*e[Q,0]*e[Q,2]+\
                                  A*dt*(u_t_y + v_t_x)*e[Q,0]*e[Q,1] + A*dt*(v_t_y + v_t_y)*e[Q,1]*e[Q,1] + A*dt*(w_t_y + v_t_z)*e[Q,2]*e[Q,1]+\
                                  A*dt*(u_t_z + w_t_x)*e[Q,0]*e[Q,2] + A*dt*(v_t_z + w_t_y)*e[Q,1]*e[Q,2] + A*dt*(w_t_z + w_t_z)*e[Q,2]*e[Q,2]
                                  )
        f[Q, :, :, :] = f[Q, :, :, :] - (f[Q, :, :, :] - f_eq[Q, :, :, :]) / tau_f

    for vvv in range(27):
        f[vvv, :, :, :] = np.roll(
            np.roll(
                np.roll(f[vvv, :, :, :], e[vvv, 0], axis=0),
                e[vvv, 1], axis=1
            ),
            e[vvv, 2], axis=2
        )

    rho = np.sum(f, axis=0)
    u[:, :, :, 0] = np.sum(f * e[:,0].reshape(27,1,1,1), axis=0) / rho
    u[:, :, :, 1] = np.sum(f * e[:,1].reshape(27,1,1,1), axis=0) / rho
    u[:, :, :, 2] = np.sum(f * e[:,2].reshape(27,1,1,1), axis=0) / rho

    u[:, 0,:, 0] = u[0, :,:, 0] = u[Nx, :,:, 0] = u[:, Ny,:, 0]= u[:, :,0, 0]= 0.0
    u[:, :, Ny, 0] = U
    u[:, 0,:, 1] = u[0, :,:, 1] = u[Nx, :,:, 1] = u[:, Ny,:, 1]= u[:, :,0, 1]= u[:, :, Ny, 1] = 0.0
    u[:, 0,:, 2] = u[0, :,:, 2] = u[Nx, :,:, 2] = u[:, Ny,:, 2]= u[:, :,0, 2]= u[:, :, Ny, 2] = 0.0

    rho[:,:, 0] = 4.0*rho[:,:,1]-6.0*rho[:,:,2] + 4.0*rho[:,:,3] - rho[:,:,4]
    rho[:,:, Nz] = 4.0*rho[:,:,Nz-1]-6.0*rho[:,:,Nz-2] + 4.0*rho[:,:,Nz-3] - rho[:,:,Nz-4]
    rho[0, :,:] = 4.0*rho[1, :,:] - 6.0*rho[2, :,:] + 4.0*rho[3, :,:] - rho[4, :,:]
    rho[Nx, :,:] = 4.0*rho[Nx-1,:,:] - 6.0*rho[Nx-2,:,:] + 4.0*rho[Nx-3,:,:] - rho[Nx-4,:,:]
    rho[:, 0,:] = 4.0*rho[:,1,:] - 6.0*rho[:,2,:] + 4.0*rho[:,3,:] - rho[:,4,:]
    rho[:, Ny,:] = 4.0*rho[:,Ny-1,:] - 6.0*rho[:,Ny-2,:] + 4.0*rho[:,Ny-3,:] - rho[:,Ny-4,:]

    error = np.sum(np.sqrt((u_n-u[:, :, :, 0])**2+(v_n-u[:, :, :, 1])**2+(w_n-u[:, :, :, 2])**2))/np.sum(np.sqrt(u[:, :, :, 0]**2+u[:, :, :, 1]**2+u[:, :, :, 2]**2))
    u_n = u[:, :, :, 0].copy() 
    v_n = u[:, :, :, 1].copy()
    w_n = u[:, :, :, 2].copy()

    # Create and run the quantum circuit for LBM, rho
    qc = QLBM(density_field=Psi_qlbm[t, :, :, :], velocity_field=u_LBM, number_velocities=NUMBER_DISCRETE_VELOCITIES)
    
    # Compile and run the quantum circuit
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit).result()
    
    # Extract and process the statevector
    statevector = np.array(result.get_statevector())
    real_part_statevector = np.real(statevector[:N_POINTS_X * N_POINTS_Y * N_POINTS_Z])
    reshaped_real_part = np.reshape(real_part_statevector, (N_POINTS_X, N_POINTS_Y, N_POINTS_Z),order='F')
    
    # Normalize and update Psi_qlbm
    norm_factor = np.linalg.norm(Psi_qlbm[t, :, :, :].flatten()) * 4 * np.sqrt(2)
    Psi_qlbm[t + 1, :, :, :] = reshaped_real_part * norm_factor
    Psi_qlbm[t + 1,:,:, 0] = 4.0*Psi_qlbm[t + 1,:,:,1]-6.0*Psi_qlbm[t + 1,:,:,2] + 4.0*Psi_qlbm[t + 1,:,:,3] - Psi_qlbm[t + 1,:,:,4]
    Psi_qlbm[t + 1,:,:, Nz] = 4.0*Psi_qlbm[t + 1,:,:,Nz-1]-6.0*Psi_qlbm[t + 1,:,:,Nz-2] + 4.0*Psi_qlbm[t + 1,:,:,Nz-3] - Psi_qlbm[t + 1,:,:,Nz-4]
    Psi_qlbm[t + 1,0, :,:] = 4.0*Psi_qlbm[t + 1,1, :,:] - 6.0*Psi_qlbm[t + 1,2, :,:] + 4.0*Psi_qlbm[t + 1,3, :,:] - Psi_qlbm[t + 1,4, :,:]
    Psi_qlbm[t + 1,Nx, :,:] = 4.0*Psi_qlbm[t + 1,Nx-1,:,:] - 6.0*Psi_qlbm[t + 1,Nx-2,:,:] + 4.0*Psi_qlbm[t + 1,Nx-3,:,:] - Psi_qlbm[t + 1,Nx-4,:,:]
    Psi_qlbm[t + 1,:, 0,:] = 4.0*Psi_qlbm[t + 1,:,1,:] - 6.0*Psi_qlbm[t + 1,:,2,:] + 4.0*Psi_qlbm[t + 1,:,3,:] - Psi_qlbm[t + 1,:,4,:]
    Psi_qlbm[t + 1,:, Ny,:] = 4.0*Psi_qlbm[t + 1,:,Ny-1,:] - 6.0*Psi_qlbm[t + 1,:,Ny-2,:] + 4.0*Psi_qlbm[t + 1,:,Ny-3,:] - Psi_qlbm[t + 1,:,Ny-4,:]
    
    # Create and run the quantum circuit for LBM, u
    qc1 = QLBM1(density_field=Psi_qlbm0, velocity_field=u_LBM, number_velocities=NUMBER_DISCRETE_VELOCITIES)
    compiled_circuit1 = transpile(qc1, simulator1)
    result1 = simulator1.run(compiled_circuit1).result()
    
    # Process the quantum statevector to update Psi_qlbm
    statevector1 = np.array(result1.get_statevector())
    real_part_statevector1 = np.real(statevector1[:N_POINTS_X * N_POINTS_Y * N_POINTS_Z])
    real_part_statevector_reshaped1 = np.reshape(real_part_statevector1, (N_POINTS_X, N_POINTS_Y, N_POINTS_Z), order='F')
    norm_factor1 = np.linalg.norm(Psi_qlbm0.flatten()) * 4 * np.sqrt(2)
    # Normalize and update the scalar field for the next timestep
    Psi_qlbm1[t + 1, :, :, :] = real_part_statevector_reshaped1 * norm_factor1 / Psi_qlbm0
    Psi_qlbm1[t + 1,:, 0,:] = Psi_qlbm1[t + 1,0, :,:] = Psi_qlbm1[t + 1,Nx, :,:] = Psi_qlbm1[t + 1,:, Ny,:]= Psi_qlbm1[t + 1,:, :,0]= 0.0
    Psi_qlbm1[t + 1,:, :, Nz] = U

    # Create and run the quantum circuit for LBM, v
    qc2 = QLBM2(density_field=Psi_qlbm0, velocity_field=u_LBM, number_velocities=NUMBER_DISCRETE_VELOCITIES)
    compiled_circuit2 = transpile(qc2, simulator2)
    result2 = simulator2.run(compiled_circuit2).result()
    
    # Process the quantum statevector to update Psi_qlbm
    statevector2 = np.array(result2.get_statevector())
    real_part_statevector2 = np.real(statevector2[:N_POINTS_X * N_POINTS_Y * N_POINTS_Z])
    real_part_statevector_reshaped2 = np.reshape(real_part_statevector2, (N_POINTS_X, N_POINTS_Y, N_POINTS_Z), order='F')
    norm_factor2 = np.linalg.norm(Psi_qlbm0.flatten()) * 4 * np.sqrt(2)
    # Normalize and update the scalar field for the next timestep
    Psi_qlbm2[t + 1, :, :, :] = real_part_statevector_reshaped2 * norm_factor2 / Psi_qlbm0
    Psi_qlbm2[t + 1,:, 0,:] = Psi_qlbm2[t + 1,0, :,:] = Psi_qlbm2[t + 1,Nx, :,:] = Psi_qlbm2[t + 1,:, Ny,:]= Psi_qlbm2[t + 1,:, :,0]= Psi_qlbm2[t + 1,:, :, Nz] = 0.0

    # Create and run the quantum circuit for LBM, v
    qc3 = QLBM3(density_field=Psi_qlbm0, velocity_field=u_LBM, number_velocities=NUMBER_DISCRETE_VELOCITIES)
    compiled_circuit3 = transpile(qc3, simulator3)
    result3 = simulator3.run(compiled_circuit3).result()
    
    # Process the quantum statevector to update Psi_qlbm
    statevector3 = np.array(result3.get_statevector())
    real_part_statevector3 = np.real(statevector3[:N_POINTS_X * N_POINTS_Y * N_POINTS_Z])
    real_part_statevector_reshaped3 = np.reshape(real_part_statevector3, (N_POINTS_X, N_POINTS_Y, N_POINTS_Z), order='F')
    norm_factor3 = np.linalg.norm(Psi_qlbm0.flatten()) * 4 * np.sqrt(2)
    # Normalize and update the scalar field for the next timestep
    Psi_qlbm3[t + 1, :, :, :] = real_part_statevector_reshaped3 * norm_factor3 / Psi_qlbm0
    Psi_qlbm3[t + 1,:, 0,:] = Psi_qlbm3[t + 1,0, :,:] = Psi_qlbm3[t + 1,Nx, :,:] = Psi_qlbm3[t + 1,:, Ny,:]= Psi_qlbm3[t + 1,:, :,0]= Psi_qlbm3[t + 1,:, :, Nz] = 0.0
    Psi_qlbm0 = Psi_qlbm[t + 1, :, :, :].copy()
    
    u_LBM[:,:, :,0] = Psi_qlbm1[t + 1, :, :, :]
    u_LBM[:,:, :,1] = Psi_qlbm2[t + 1, :, :, :]
    u_LBM[:,:, :,2] = Psi_qlbm3[t + 1, :, :, :]
    error1 = np.sum(np.sqrt((Psi_qlbm1[t + 1, :, :, :]-Psi_qlbm1[t, :, :, :])**2+(Psi_qlbm2[t + 1, :, :, :]-Psi_qlbm2[t, :, :, :])**2+(Psi_qlbm3[t + 1, :, :, :]-Psi_qlbm3[t, :, :, :])**2))/np.sum(np.sqrt(Psi_qlbm1[t + 1, :, :, :]**2+Psi_qlbm2[t + 1, :, :, :]**2+Psi_qlbm3[t + 1, :, :, :]**2))
    error2 = np.sum(np.sqrt((Psi_qlbm1[t + 1, :, :, :]-u_n)**2+(Psi_qlbm2[t + 1, :, :, :]-v_n)**2+(Psi_qlbm3[t + 1, :, :, :]-w_n)**2))/np.sum(np.sqrt(u_n**2+v_n**2+w_n**2))
    print(t+1,error,error1,error2)

    q_error.append(error1)
    c_error.append(error)
    qc_error.append(error2)

    if error1 < 1e-6:
        break

1 1.0 0.10695890262410633 2.4297464659959186e-05
2 0.06765137322061944 0.06765719208492248 9.806307180615729e-05
3 0.05065752174861489 0.050664060558672 0.00012947009045791372
4 0.04558668984399281 0.04561425476993328 0.0002695923244336499


KeyboardInterrupt: 

In [ ]:
print(t+1,error,error1,error2)

159
0.00033471813797722244
0.0003347181379771898
0.00033426482610025577
0.00033426482610002494


In [ ]:
np.savetxt("Re=100_32\QLBM_u_"+str(t)+".csv",Psi_qlbm1[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32\QLBM_v_"+str(t)+".csv",Psi_qlbm2[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32\QLBM_w_"+str(t)+".csv",Psi_qlbm3[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32\QLBM_rho_"+str(t)+".csv",Psi_qlbm[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32\CLBM_u_"+str(t)+".csv",u[:,:,:,0].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32\CLBM_v_"+str(t)+".csv",u[:,:,:,1].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32\CLBM_w_"+str(t)+".csv",u[:,:,:,2].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32\CLBM_rho_"+str(t)+".csv",rho.reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")

In [ ]:
x = np.linspace(0,1,Nx+1)[:,None]
y = np.linspace(0,1,Ny+1)[:,None]
X, Y = np.meshgrid(x, y)
fig = plt.figure(figsize=(10, 8))
plt.streamplot(X,Y,u[:,int(Nx/2),:,0].T,u[:,int(Nx/2),:,2].T,density=3)
plt.tick_params(labelsize=15)
plt.xlim(0.0,1.0)
plt.xlabel('$x$', size = 18)
plt.ylabel('$y$', size = 18)
plt.title(r"$Re=100,N = 32 \times 32 \times 32$", size = 18)
plt.ylim(0.0,1.0)
fig1=plt.gcf()
ax=plt.gca();#获得坐标轴的句柄
ax.spines['bottom'].set_linewidth(1)###设置底部坐标轴的粗细
ax.spines['left'].set_linewidth(1)####设置左边坐标轴的粗细
ax.spines['right'].set_linewidth(1)###设置右边坐标轴的粗细
ax.spines['top'].set_linewidth(1)
#plt.savefig('streamline_100_32_classical.jpg', format='jpg', dpi=100)
plt.show()

In [ ]:
x = np.linspace(0,1,Nx+1)[:,None]
y = np.linspace(0,1,Ny+1)[:,None]
X, Y = np.meshgrid(x, y)
fig = plt.figure(figsize=(10, 8))
plt.streamplot(X,Y,Psi_qlbm1[:,int(Nx/2),:].T,Psi_qlbm2[:,int(Nx/2),:].T,density=3)
plt.tick_params(labelsize=15)
plt.xlim(0.0,1.0)
plt.xlabel('$x$', size = 18)
plt.ylabel('$y$', size = 18)
plt.title(r"$Re=100,N = 32 \times 32 \times 32$", size = 18)
plt.ylim(0.0,1.0)
fig1=plt.gcf()
ax=plt.gca();#获得坐标轴的句柄
ax.spines['bottom'].set_linewidth(1)###设置底部坐标轴的粗细
ax.spines['left'].set_linewidth(1)####设置左边坐标轴的粗细
ax.spines['right'].set_linewidth(1)###设置右边坐标轴的粗细
ax.spines['top'].set_linewidth(1)
#plt.savefig('streamline_100_32_quantum.jpg', format='jpg', dpi=100)
plt.show()